# Preprocessing: text to training batches

**The whole idea in one sentence:** glue every document into one enormous list of
numbers, write it to disk, and let training grab random slices of it.

Everything else in this notebook is a detail of doing that without silently
corrupting your data — and the corruption is the interesting part, because none
of it raises an exception. A wrapped token id, a stale file, a target that leaks
the answer: each one produces a model that trains fine and is quietly wrong.

By the end you'll have:

- built the whole pipeline by hand on three documents you can read
- seen why targets are the inputs *shifted by one*, and watched the unshifted
  version produce a **better-looking** loss
- overflowed a `uint16` on purpose and watched Chinese text come back as English
- read a file with the wrong dtype and gotten plausible garbage
- worked out what sharding costs you, and why it is ~0.001%
- run the packaged version and measured it

The first seven sections build everything from scratch. The last one shows the
version in `litterbox.data` that does the same thing.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.set_printoptions(linewidth=100)

---
## 1. Four stages

```
source        ->  tokenize  ->  pack            ->  sample
Iterable[str]     ids           flat array on disk   [B, S] batches
```

Let's do all four by hand. Three documents, and a byte tokenizer so the ids stay
readable — every id below is literally the UTF-8 byte value.

In [ ]:
docs = [
    "the cat sat",
    "a dog ran",
    "birds fly high",
]

def encode(s):
    return list(s.encode("utf-8"))

def decode(ids):
    return bytes(i for i in ids if i < 256).decode("utf-8", errors="replace")

for d in docs:
    print(f"{d!r:20} -> {encode(d)}")

### Glue them together

Documents are concatenated into **one continuous stream**, with a separator token
between them so the model gets a signal that one thing ended and another began.

Real tokenizers have an end-of-text id for this. The byte tokenizer has none, so
we'll use `256` as a stand-in and remember that our vocabulary is now 257.

In [ ]:
EOS = 256

stream = []
doc_starts = []          # where each document begins, in the global stream
for d in docs:
    doc_starts.append(len(stream))
    stream.extend(encode(d))
    stream.append(EOS)

stream = np.array(stream)
print(f"{len(stream)} tokens from {len(docs)} documents")
print(stream)
print(f"\ndocument starts: {doc_starts}")

That array is the entire dataset. Not a list of documents — **one flat sequence**.
The document boundaries still exist (we recorded them), but nothing in the array
itself marks them apart from those `256`s.

---
## 2. Sampling windows

A training example is just a slice at a random offset. Pick a start, take
`seq_len` tokens.

In [ ]:
seq_len = 8
rng = np.random.default_rng(0)

for _ in range(4):
    start = rng.integers(0, len(stream) - seq_len)
    window = stream[start : start + seq_len]
    print(f"start {start:>2}: {window}  {decode(window.tolist())!r}")

Look at the last one or two: some windows straddle a document boundary — they
contain the tail of one document, an EOS, and the head of the next.

**That is fine, and it is what everyone does.** The model is learning "predict
the next token", and a fragment is still valid text. The EOS teaches it that
documents end. Refusing to split would mean padding, which wastes a large
fraction of your compute on nothing.

(SFT is the opposite: there an example is atomic and splitting it is destructive.
Different problem, different pipeline.)

---
## 3. Targets: the trap that makes loss look *better*

The model reads `x` and must predict the **next** token, so targets are the
inputs shifted one position forward:

```
x = stream[i     : i + S]
y = stream[i + 1 : i + 1 + S]
```

In [ ]:
i = 0
x = stream[i : i + seq_len]
y = stream[i + 1 : i + 1 + seq_len]

print("x:", x, f" {decode(x.tolist())!r}")
print("y:", y, f" {decode(y.tolist())!r}")
print()
for a, b in zip(x.tolist(), y.tolist()):
    print(f"  see {a:>3} {decode([a])!r:5} -> predict {b:>3} {decode([b])!r}")

### Now get it wrong on purpose

Suppose you pass `y = x` instead — targets not shifted. The model is asked to
predict the token it is *currently looking at*.

You might expect that to fail loudly. It does the opposite. Modern models **tie**
the embedding and output weights, so the final hidden state still carries the
input token's embedding, and `h · embed[token]` is large for exactly that token.
Predicting the current token is nearly free.

Here is a stand-in for a transformer's last layer: a residual stream that starts
as the token embedding, has some amount added to it by the layers, gets
normalized, and is projected back through the tied embedding.

In [ ]:
import math

torch.manual_seed(0)
V, D = 512, 128
embed = torch.nn.Embedding(V, D)
torch.nn.init.normal_(embed.weight, std=0.02)
tokens = torch.randint(0, V, (32, 129))

print(f"an untrained model must score ln(vocab) = {math.log(V):.3f}\n")
print(f"{'signal the layers added':>24} {'shifted y':>11} {'unshifted y':>13}")
print("-" * 52)

for scale in (0.0, 1.0, 3.0, 10.0):
    x_ = tokens[:, :-1]
    h = embed(x_)                                    # residual stream: the embedding
    h = h + torch.randn_like(h) * 0.02 * scale       # ...plus what the layers wrote
    h = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + 1e-5)   # final norm
    logits = h @ embed.weight.T                      # tied output projection

    good = F.cross_entropy(logits.reshape(-1, V), tokens[:, 1:].reshape(-1)).item()
    bad  = F.cross_entropy(logits.reshape(-1, V), x_.reshape(-1)).item()
    print(f"{scale:>23.0f}x {good:>11.3f} {bad:>13.3f}")

Read the `1x` row. The correct setup sits at **6.27**, right where an untrained
model belongs. The broken one reports **4.47** — dramatically better, and
completely meaningless.

Two things make this genuinely dangerous:

1. **It looks like good news.** A lower loss is what you were hoping for. Nobody
   investigates a number that is better than expected.
2. **It is worst at initialisation.** The leak shrinks as the layers write more
   into the residual stream — so it is largest at step 0, exactly when you first
   glance at the loss and decide things are working.

**The check:** a freshly initialised model must score `ln(vocab_size)`. Anything
meaningfully below it means your targets are leaking. One assertion, and it
catches this permanently.

In [ ]:
def assert_init_loss_is_random(loss: float, vocab_size: int, tol: float = 0.15):
    expected = math.log(vocab_size)
    if abs(loss - expected) > tol:
        raise AssertionError(
            f"init loss {loss:.3f} vs ln(vocab)={expected:.3f}. "
            f"{'targets may be leaking' if loss < expected else 'init may be wrong'}"
        )
    print(f"init loss {loss:.3f} ~ ln(vocab) {expected:.3f}  OK")

assert_init_loss_is_random(6.27, 512)
try:
    assert_init_loss_is_random(4.47, 512)
except AssertionError as e:
    print("caught:", e)

---
## 4. Choosing a dtype (this is a correctness decision)

You are about to write billions of integers to disk. Each one needs a size, and
the size fixes the largest value you can store:

| dtype | bytes each | largest value |
|---|---|---|
| `uint16` | 2 | **65,535** |
| `uint32` | 4 | 4,294,967,295 |

So the question is only: **does the largest token id fit in 65,535?** That is
`vocab_size`, which means the tokenizer decides the dtype — and the file being
half the size is a *consequence*, never the reason.

In [ ]:
def choose_dtype(vocab_size):
    if vocab_size <= 2**16:
        return np.uint16
    if vocab_size <= 2**32:
        return np.uint32
    raise ValueError("vocab too large")

for name, v in [("Llama-2 / Mistral", 32_000), ("GPT-2", 50_257), ("GPT-4 cl100k", 100_277),
                ("Llama-3", 128_256), ("GPT-4o o200k", 200_019), ("Gemma", 256_000)]:
    dt = choose_dtype(v)
    fits = "fits" if dt is np.uint16 else "TOO BIG"
    print(f"{name:<20} {v:>8,}   {np.dtype(dt).name:<7} {fits}")

GPT-2 fits, which is why so much tutorial code hardcodes `uint16` and gets away
with it. Most modern tokenizers do not.

### What "too big" actually does

Not an error. Let's overflow one on purpose.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    text = "研究人员发表了关于分词效率的研究成果。"
    ids = enc.encode(text)
    print(f"vocab {enc.n_vocab:,}, largest id here: {max(ids):,}  (uint16 max is 65,535)\n")

    stored = np.array(ids, dtype=np.int64).astype(np.uint16)   # the realistic mistake
    for a, b in zip(ids, stored.tolist()):
        if a != b:
            print(f"  {a:>7,} -> {b:>6,}      ({a} mod 65536)")

    print(f"\n  original : {text!r}")
    print(f"  recovered: {enc.decode(stored.tolist())!r}")
    print("\nNo exception. No warning. Just different text.")
except ImportError:
    print("pip install tiktoken to run this cell")
    print("(the id 157,422 becomes 157,422 - 2*65,536 = 26,350 — a valid, unrelated token)")

`uint16` holds 65,536 distinct values, so anything larger wraps around like an
odometer. Token 157,422 becomes 26,350 — a perfectly valid id for a completely
different token.

And note **which** text broke. BPE assigns low ids to frequent tokens, so common
English never overflows. The bug hides during your ASCII testing and appears the
moment you feed it another language, or code, or anything rare.

---
## 5. A `.bin` is just bytes

The file has no header, no schema, no idea how it was written. The dtype is
purely *your* instruction for where to put the cuts.

In [ ]:
import os
import tempfile

path = os.path.join(tempfile.mkdtemp(), "toy.bin")
np.array([72, 101, 108, 108, 111, 33], dtype=np.uint16).tofile(path)

print(f"wrote 6 ids as uint16 -> {os.path.getsize(path)} bytes")
print(f"raw bytes: {open(path, 'rb').read()}\n")
print("the SAME bytes, read two ways:")
print("  as uint16 ->", np.memmap(path, dtype=np.uint16, mode="r").tolist(), " correct")
print("  as uint32 ->", np.memmap(path, dtype=np.uint32, mode="r").tolist(), "       garbage")

Half as many tokens, all nonsense, no error. Which is the entire argument for
writing a metadata sidecar — we'll do that in section 7.

---
## 6. Sharding

Instead of one enormous file, write many smaller ones:

```
train.bin  200 GB          train_0000.bin  200 MB
                           train_0001.bin  200 MB
                           ...
```

Same tokens, same order, chopped up. The cut points **mean nothing** — the last
token of shard 0 and the first of shard 1 were adjacent in the original text.

Three reasons, and *none of them is read speed*:

1. **Memory while packing.** You buffer ids before writing. One shard's worth at
   a time keeps memory flat no matter how large the corpus.
2. **Crash recovery.** Tokenizing can take hours. Completed shards survive.
3. **Convenience.** Copy three shards to test on. Delete half.

### The one wrinkle

A window can land across a seam.

In [ ]:
shard_size, seq_len_demo = 1000, 200

for start in (300, 950, 2100):
    shard, offset = divmod(start, shard_size)
    end = offset + seq_len_demo
    if end <= shard_size:
        print(f"  start {start:>4} -> shard {shard}, offset {offset:>3}..{end:<4} fine")
    else:
        print(f"  start {start:>4} -> shard {shard}, offset {offset:>3}..1000 then "
              f"{end - shard_size} more from shard {shard + 1}   STRADDLES")

bad = seq_len_demo - 1
print(f"\n  toy sizes:  {bad}/{shard_size} starts straddle = {bad / shard_size:.1%}")
print(f"  real sizes: 1023/100,000,000 = {1023 / 100_000_000:.5%}")

20% at toy sizes, **0.001%** at real ones — a 1024-token window against a
100M-token shard is a hair against a football field.

So the fix is to simply not sample those. You lose one start position in 100,000
and the loader stays a single memmap slice with no stitching logic.

---
## 7. The sidecar

We now have everything needed to write the real thing — and one file that makes
the rest readable.

A `.bin` cannot tell you which tokenizer produced it. Swap tokenizers, reuse a
stale file, and you train on noise with **no error anywhere** — just a loss curve
you blame on your architecture.

In [ ]:
import json
from pathlib import Path

def pack(docs, encode_fn, out_dir, *, vocab_size, eos_id, tokenizer_name,
         shard_tokens=700):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dtype = np.dtype(choose_dtype(vocab_size))

    buf = np.empty(shard_tokens, dtype=dtype)
    n, shards, doc_starts, total = 0, [], [], 0

    def flush():
        nonlocal n
        if n == 0:
            return
        name = f"train_{len(shards):04d}.bin"
        buf[:n].tofile(out_dir / name)
        shards.append({"file": name, "n_tokens": int(n)})
        n = 0

    for doc in docs:
        ids = encode_fn(doc) + ([eos_id] if eos_id is not None else [])
        assert max(ids) < vocab_size, f"id {max(ids)} would wrap in {dtype}"
        doc_starts.append(total)
        total += len(ids)

        pos = 0
        while pos < len(ids):
            take = min(len(buf) - n, len(ids) - pos)
            buf[n : n + take] = ids[pos : pos + take]
            n += take
            pos += take
            if n == len(buf):
                flush()
    flush()

    meta = {"tokenizer_name": tokenizer_name, "vocab_size": vocab_size,
            "dtype": dtype.name, "eos_id": eos_id, "shards": shards,
            "n_tokens": total, "n_documents": len(doc_starts)}
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    np.array(doc_starts, dtype=np.int64).tofile(out_dir / "train_docs.bin")
    return meta


big = [f"Document {i}. the cat sat on the mat. " * 2 for i in range(40)]
d = Path(tempfile.mkdtemp())
meta = pack(big, encode, d, vocab_size=257, eos_id=EOS, tokenizer_name="byte")

print(json.dumps({k: v for k, v in meta.items() if k != "shards"}, indent=2))
print(f"\n{len(meta['shards'])} shards:", [s["file"] for s in meta["shards"]][:4], "...")

Now the reader, with the check that makes all of this worth it.

In [ ]:
class Dataset:
    def __init__(self, path, seq_len, tokenizer_name=None):
        self.path = Path(path)
        self.seq_len = seq_len
        self.meta = json.loads((self.path / "meta.json").read_text())

        if tokenizer_name is not None and tokenizer_name != self.meta["tokenizer_name"]:
            raise ValueError(
                f"shards were packed with {self.meta['tokenizer_name']!r} but you "
                f"passed {tokenizer_name!r} - the ids mean something different"
            )

        dt = np.dtype(self.meta["dtype"])
        self.shards = [np.memmap(self.path / s["file"], dtype=dt, mode="r")
                       for s in self.meta["shards"]]
        # only positions where a whole window fits inside one shard
        self.valid = np.array([max(0, len(s) - seq_len) for s in self.shards])
        self.cum = np.cumsum(self.valid)

    def get_batch(self, batch_size, step, seed=0):
        rng = np.random.default_rng([seed, step])       # NOT a stateful generator
        flat = rng.integers(0, self.cum[-1], size=batch_size)
        si = np.searchsorted(self.cum, flat, side="right")
        off = flat - (self.cum[si] - self.valid[si])

        w = np.stack([self.shards[s][o : o + self.seq_len + 1]
                      for s, o in zip(si, off)]).astype(np.int64)
        return torch.from_numpy(w[:, :-1]), torch.from_numpy(w[:, 1:])


ds = Dataset(d, seq_len=16)
x, y = ds.get_batch(4, step=0)
print(f"x {tuple(x.shape)}  y {tuple(y.shape)}")
print("targets shifted by one:", bool((x[:, 1:] == y[:, :-1]).all()))
print()
print("the stale-shard check:")
try:
    Dataset(d, seq_len=16, tokenizer_name="tiktoken:gpt2")
except ValueError as e:
    print(f"  ValueError: {e}")

### Why `default_rng([seed, step])`

Batches are derived from `(seed, step)` rather than from a generator you keep
advancing. That means step 5,000 gives the same batch whether you reached it in
one run or resumed from a checkpoint — a stateful RNG would silently hand a
resumed run different data, and your experiment records would stop being
reproducible.

In [ ]:
a = ds.get_batch(4, step=99)[0]
b = ds.get_batch(4, step=99)[0]
c = ds.get_batch(4, step=100)[0]

print("same step, twice ->  identical:", bool((a == b).all()))
print("next step        ->  different:", not bool((a == c).all()))

fresh = Dataset(d, seq_len=16)
for s in range(99):
    fresh.get_batch(4, step=s)
print("after 99 other calls, step 99 is still:", bool((fresh.get_batch(4, step=99)[0] == a).all()))

---
## 8. The packaged version

Everything above, hardened, is in `litterbox.data`. Same design; it also handles
documents longer than a shard, multiple splits, worker processes, and truncated
files.

In [ ]:
from litterbox.data import PackedDataset, pack as lb_pack
from litterbox.data.tokenizer import build_tokenizer

cfg = {"type": "tiktoken", "encoding": "gpt2"}
try:
    tok = build_tokenizer(cfg)
except ImportError:
    cfg = {"type": "byte"}
    tok = build_tokenizer(cfg)
    print("tiktoken not installed - using the byte tokenizer\n")

corpus = [("The researchers examined how tokenization affects model quality. " * 6)
          + f" Document {i}." for i in range(3000)]

out = Path(tempfile.mkdtemp())
m = lb_pack(corpus, tok, out, tokenizer_cfg=cfg, shard_tokens=100_000)

real = PackedDataset(out, seq_len=256, tokenizer=tok)
print(real)
print(f"{real.n_valid_positions:,} valid window starts")
print(f"\ndecoded from the stream: {tok.decode(real.read(0, 18).tolist())!r}")

In [ ]:
import time

for bs, sl in ((16, 256), (32, 1024)):
    ds_ = PackedDataset(out, seq_len=sl)
    for s in range(20):
        ds_.get_batch(bs, step=s)
    t0 = time.perf_counter()
    N = 200
    for s in range(N):
        ds_.get_batch(bs, step=1000 + s)
    dt = (time.perf_counter() - t0) / N
    print(f"B={bs:<3} S={sl:<5}  {dt * 1e6:>6.0f} us/batch   {bs * sl / dt / 1e6:>7.1f} Mtok/s")

print("\nA forward+backward on a ~100M model is tens of milliseconds.")
print("Loading is microseconds - which is why this is a memmap slice,")
print("not a torch DataLoader with worker processes.")

---
## What to remember

Four failures, all silent, each with a cheap defence:

| failure | defence |
|---|---|
| targets not shifted | assert init loss ≈ `ln(vocab_size)` |
| id too large for the dtype | derive dtype from `vocab_size`, assert `max(ids) < vocab_size` |
| reading with the wrong dtype | record it in `meta.json` |
| stale shards from another tokenizer | record the tokenizer, check it on load |

None of them raise on their own. That is the whole reason the checks exist.

---

## Exercises

1. **Prove it end to end.** Decode the entire packed stream and assert every
   source document appears in it, exactly once.
2. **Break the sidecar.** Truncate a `.bin` by a few bytes and make `Dataset`
   detect it. (Hint: compare file size against `n_tokens * itemsize`.)
3. **Length curriculum.** Make `seq_len` a function of `step` — 256 for the first
   1,000 steps, then 512, then 1,024 — keeping `batch_size * seq_len` constant.
   This is why `get_batch` takes `step` at all.
4. **Document masking.** Using `train_docs.bin`, build a per-token document id
   for a batch, then the block-diagonal mask that stops a window attending across
   a document boundary.
5. **Measure the split rate.** What fraction of sampled windows straddle a
   document boundary at `seq_len=1024`? How does it change with document length?

---

## Next

- [`demo/tokenizers/bpe.ipynb`](../tokenizers/bpe.ipynb) — where the ids come from
- `examples/data_pipeline.py` — the same flow as a script, with benchmarks
- [`DEFERRED.md`](../../DEFERRED.md) — what this pipeline deliberately does not do yet